# Medium Analyzer: Ingestion Pipeline Notebook

This notebook mirrors the ingestion workflow from `ingestion.py` and explains each step.

Pipeline stages:
1. Load environment variables
2. Load source document
3. Split text into chunks
4. Build embeddings and ingest to Pinecone


In [1]:
from pathlib import Path
from dotenv import load_dotenv

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'ingestion.py').exists():
    PROJECT_DIR = Path('/Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer')

load_dotenv(PROJECT_DIR / '.env')
print('SECTION 0: Environment loaded from .env')
print(f'Project directory: {PROJECT_DIR}')

SECTION 0: Environment loaded from .env
Project directory: /Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer


## Section 1: Verify Required Configuration

Before ingestion, validate that all required environment variables are present.

In [2]:
import os

required_keys = [
    'OPENAI_API_KEY',
    'LANGSMITH_API_KEY',
    'LANGSMITH_PROJECT',
    'LANGSMITH_TRACING',
    'INDEX_NAME',
    'PINECONE_API_KEY',
]

print('SECTION 1 RESULT:')
for key in required_keys:
    value = os.getenv(key)
    status = 'OK' if value else 'MISSING'
    print(f'- {key}: {status}')

SECTION 1 RESULT:
- OPENAI_API_KEY: OK
- LANGSMITH_API_KEY: OK
- LANGSMITH_PROJECT: OK
- LANGSMITH_TRACING: OK
- INDEX_NAME: OK
- PINECONE_API_KEY: OK


## Section 2: Load Document

The `TextLoader` reads `mediumblog1.txt` and converts it into LangChain Document objects.

In [3]:
from langchain_community.document_loaders import TextLoader

source_path = PROJECT_DIR / 'mediumblog1.txt'
loader = TextLoader(str(source_path))
documents = loader.load()

print('SECTION 2 RESULT:')
print(f'- Source path: {source_path}')
print(f'- Document count: {len(documents)}')
print(f"- First 180 chars: {documents[0].page_content[:180].replace(chr(10), ' ')}...")

SECTION 2 RESULT:
- Source path: /Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer/mediumblog1.txt
- Document count: 1
- First 180 chars:  Search Write  Emarco Vector Database: What is it and why you should know it? Ejiro Onose Ejiro Onose  · Follow  9 min read · Dec 22, 2023 100        “If 2021 was the year of graph...


## Section 3: Split Into Chunks

The splitter creates fixed-size chunks (`chunk_size=1000`, `chunk_overlap=0`) for embedding and indexing.

In [4]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks = splitter.split_documents(documents)

print('SECTION 3 RESULT:')
print(f'- Chunk count: {len(chunks)}')
if chunks:
    print(f'- First chunk length: {len(chunks[0].page_content)}')
    print(f"- First chunk preview: {chunks[0].page_content[:180].replace(chr(10), ' ')}...")

Created a chunk of size 1180, which is longer than the specified 1000


Created a chunk of size 1058, which is longer than the specified 1000


SECTION 3 RESULT:
- Chunk count: 20
- First chunk length: 954
- First chunk preview: Search Write  Emarco Vector Database: What is it and why you should know it? Ejiro Onose Ejiro Onose  · Follow  9 min read · Dec 22, 2023 100  “If 2021 was the year of graph databa...


## Section 4: Pipeline Dry Run (Recommended First)

This calls `run_ingestion(..., dry_run=True)` from `ingestion.py` to verify the full workflow without external API calls.

In [5]:
from ingestion import run_ingestion

summary = run_ingestion(source_path=source_path, dry_run=True)
print('SECTION 4 RESULT:')
print(summary)

/Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Created a chunk of size 1180, which is longer than the specified 1000


Created a chunk of size 1058, which is longer than the specified 1000


SECTION 1: Load source document
Loaded 1 document(s) from /Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer/mediumblog1.txt

SECTION 2: Split document into chunks
Created 20 chunk(s)
First chunk preview: Search Write  Emarco Vector Database: What is it and why you should know it? Ejiro Onose Ejiro Onose  · Follow  9 min read · Dec 22, 2023 100  “If 2021 was the year of graph databa...

SECTION 3: Dry run mode enabled - skipping embedding + Pinecone ingest
SECTION 4 RESULT:
{'source_path': '/Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer/mediumblog1.txt', 'document_count': 1, 'chunk_count': 20, 'ingested': False}


## Section 5: Real Pinecone Ingestion (Explicit Pinecone Call)

This section performs the direct Pinecone write call from the notebook:
- initialize `OpenAIEmbeddings`
- call `PineconeVectorStore.from_documents(chunks, embeddings, index_name=...)`

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

import os

try:
    print('SECTION 5: Creating embeddings client...')
    embeddings = OpenAIEmbeddings(openai_api_key=os.environ.get('OPENAI_API_KEY'))
    print('SECTION 5: Writing chunks to Pinecone...')
    PineconeVectorStore.from_documents(
        chunks,
        embeddings,
        index_name=os.environ['INDEX_NAME'],
    )
    print(f"SECTION 5 RESULT: Ingested {len(chunks)} chunk(s) into index '{os.environ['INDEX_NAME']}'")
except Exception as e:
    print(f"SECTION 5 RESULT: FAILED -> {type(e).__name__}: {e}")

SECTION 5: Creating embeddings client...
SECTION 5: Writing chunks to Pinecone...


SECTION 5 RESULT: Ingested 20 chunk(s) into index 'medium-analyer'


## Workflow Summary

- `ingestion.py` is the script entrypoint for CLI-style ingestion.
- This notebook is the guided, explainable workflow version.
- Both use the same logical steps: load -> split -> embed -> Pinecone upsert.
- Use dry-run first, then execute real ingestion when env and chunking look correct.